"""
preprocess_road.py
==================
전국자전거도로표준데이터.csv → 서울+경기 재전처리 → road_clean_v2.csv

변경사항 (v2):
  - 필터 기준: 시도명 → 시군구명 기반 (더 세밀한 지역 분류)
  - PK: 노선명+시군구명 복합키 (노선명 단독 PK 불가 확인)
  - 좌표 결측 행에 기점지번주소 보관 (후속 geocoding 대비)
  - 파생 피처: is_wide_road, safety_index (너비×0.7 + 길이×0.3)
  - 모델 3개 비교: LinearRegression / PolynomialRegression / RandomForest

출력:
  kride-project/data/raw_ml/road_clean_v2.csv
"""

공공체에서 휴면명조체나 맑은고딕체를 자주 사용합니다. 제목은 휴면명조체로 과제 작성하기, 내용은 맑은고딕체 작성하기 

폰트, 문서형식 , 글자크기, 표 양식 공공기관의 양식에 맞게 하기 (수업시간에 언급했으므로 중요한 사항인거같음)
이미지 > 본문과 배치, 글자처럼 취급, 가독성있게 보고서 작성하기

네모박스는 복사해서 다음페이지로 넘기기, 중간에 걸치지 말고 다음페이지로 작성

In [1]:

import os
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")   # GUI 없는 환경에서 그래프 저장용

In [2]:
# 그래프 저장용
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")


In [9]:

# ── 경로 설정 ──────────────────────────────────────────────────────────────
# Jupyter에서는 __file__이 없으므로 fallback 처리
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Jupyter 실행 시: 현재 작업 디렉토리 기준 kride-project 폴더
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "kride-project"))
    if not os.path.exists(BASE_DIR):
        BASE_DIR = os.getcwd()  # kride-project 안에서 실행하는 경우
RAW_DIR    = os.path.join(BASE_DIR, "data", "raw_ml")
INPUT_PATH = os.path.join(RAW_DIR, "전국자전거도로표준데이터.csv")
OUTPUT_PATH = os.path.join(RAW_DIR, "road_clean_v2.csv")

# 서울+경기 시군구명 목록 (시도명 대신 시군구명으로 필터)
SEOUL_GU = [
    "종로구","중구","용산구","성동구","광진구","동대문구","중랑구","성북구",
    "강북구","도봉구","노원구","은평구","서대문구","마포구","양천구","강서구",
    "구로구","금천구","영등포구","동작구","관악구","서초구","강남구","송파구","강동구",
]
GYEONGGI_SI_GUN = [
    "수원시","성남시","의정부시","안양시","부천시","광명시","평택시","동두천시",
    "안산시","고양시","과천시","구리시","남양주시","오산시","시흥시","군포시",
    "의왕시","하남시","용인시","파주시","이천시","안성시","김포시","화성시",
    "광주시","양주시","포천시","여주시","연천군","가평군","양평군",
]
TARGET_SIGUNGU = SEOUL_GU + GYEONGGI_SI_GUN


In [10]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: 원본 로드
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1: 원본 CSV 로드")
print("=" * 60)

try:
    df = pd.read_csv(INPUT_PATH, encoding="cp949")
except UnicodeDecodeError:
    df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
print(f"  원본 shape : {df.shape}")
print(f"  전체 컬럼 :")
for c in df.columns:
    print(f"    - {repr(c)}")
print()

# 노선명 PK 가능 여부 확인
n_unique = df["노선명"].nunique()
print(f"  노선명 유일값: {n_unique:,} / 전체: {len(df):,}")
print(f"  → 노선명 단독 PK {'가능' if n_unique == len(df) else '불가 (복합키 사용)'}\n")



STEP 1: 원본 CSV 로드
  원본 shape : (20262, 23)
  전체 컬럼 :
    - '노선명'
    - '노선번호'
    - '시도명'
    - '시군구명'
    - '기점도로명주소'
    - '기점지번주소'
    - '종점도로명주소'
    - '종점지번주소'
    - '기점위도'
    - '기점경도'
    - '종점위도'
    - '종점경도'
    - '주요경유지'
    - '총길이(km)'
    - '일반도로너비(m)'
    - '자전거도로너비(m)'
    - '자전거도로종류'
    - '자전거도로고시유무'
    - '관리기관명'
    - '관리기관전화번호'
    - '데이터기준일자'
    - '제공기관코드'
    - '제공기관명'

  노선명 유일값: 17,544 / 전체: 20,262
  → 노선명 단독 PK 불가 (복합키 사용)



In [11]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: 서울+경기 필터링 (시군구명 기준)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 2: 서울+경기 필터링 (시군구명 기준)")
print("=" * 60)

df_target = df[df["시군구명"].isin(TARGET_SIGUNGU)].copy()
print(f"  전체 {len(df):,}행  →  서울+경기 {len(df_target):,}행")
print(f"  시군구별 분포:\n{df_target['시군구명'].value_counts().head(15).to_string()}\n")

# 복합 PK 생성
df_target["road_id"] = df_target["노선명"] + "_" + df_target["시군구명"]
print(f"  복합키(road_id) 유일값: {df_target['road_id'].nunique():,}\n")


STEP 2: 서울+경기 필터링 (시군구명 기준)
  전체 20,262행  →  서울+경기 4,998행
  시군구별 분포:
시군구명
용인시     687
평택시     666
수원시     637
화성시     416
부천시     397
중구      321
남양주시    309
성남시     265
김포시     209
파주시     167
이천시     143
강서구      99
양천구      74
광주시      68
안성시      57

  복합키(road_id) 유일값: 3,643



In [12]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: 컬럼 선택 및 이름 정리
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 3: 컬럼 선택 및 타입 변환")
print("=" * 60)

# 원본 컬럼명 → 영문 매핑
col_map = {
    "노선명"              : "route_name",
    "시도명"              : "sido",
    "시군구명"            : "sigungu",
    "기점지번주소"        : "start_addr_jibun",      # geocoding 대비 보존
    "기점도로명주소"      : "start_addr_road",
    "기점위도"            : "start_lat",
    "기점경도"            : "start_lon",
    "종점위도"            : "end_lat",
    "종점경도"            : "end_lon",
    "총길이(km)"          : "length_km",
    "자전거전용도로너비(m)": "width_m",
    "자전거전용도로종류"  : "road_type",
    "자전거전용도로이용가능여부": "is_official",
}

# 실제 존재하는 컬럼만 선택
existing_cols = {k: v for k, v in col_map.items() if k in df_target.columns}
df_clean = df_target[list(existing_cols.keys()) + ["road_id"]].rename(columns=existing_cols)

# width_m: 자전거전용도로너비가 없으면 일반도로너비(m)로 대체
if "width_m" not in df_clean.columns:
    alt_width = "일반도로너비(m)"
    if alt_width in df_target.columns:
        df_clean["width_m"] = df_target[alt_width].values
        print(f"  ⚠️  자전거전용도로너비 없음 → '{alt_width}' 대체 사용")
    else:
        df_clean["width_m"] = float("nan")
        print(f"  ⚠️  너비 컬럼 없음 → width_m = NaN")

# 숫자 변환 (존재하는 컬럼만)
for col in ["width_m", "length_km", "start_lat", "start_lon", "end_lat", "end_lon"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print(f"  정제 후 shape : {df_clean.shape}")
print(f"  결측값 현황:\n{df_clean.isnull().sum().to_string()}\n")



STEP 3: 컬럼 선택 및 타입 변환
  ⚠️  자전거전용도로너비 없음 → '일반도로너비(m)' 대체 사용
  정제 후 shape : (4998, 12)
  결측값 현황:
route_name             0
sido                   0
sigungu                0
start_addr_jibun    3059
start_addr_road     1754
start_lat           3247
start_lon           3247
end_lat             3271
end_lon             3271
length_km              0
road_id                0
width_m             4635



In [13]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: 파생 피처 생성
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 4: 파생 피처 생성")
print("=" * 60)

# is_wide_road: 너비 2.0m 이상 → 1 (width_m 없으면 0)
if "width_m" in df_clean.columns:
    df_clean["is_wide_road"] = (df_clean["width_m"] >= 2.0).astype(int)
else:
    df_clean["is_wide_road"] = 0

# is_official: 이용가능여부 → 1/0
if "is_official" in df_clean.columns:
    df_clean["is_official"] = df_clean["is_official"].apply(
        lambda x: 1 if str(x).strip() in ["가능", "Y", "1", "True"] else 0
    )

# safety_index: 너비(0.7) + 길이(0.3) → MinMaxScaler 정규화 후 가중합
valid_mask = df_clean[["width_m", "length_km"]].notna().all(axis=1)
scaler_si = MinMaxScaler()
normalized = scaler_si.fit_transform(df_clean.loc[valid_mask, ["width_m", "length_km"]])
df_clean.loc[valid_mask, "safety_index"] = normalized[:, 0] * 0.7 + normalized[:, 1] * 0.3

wide = df_clean["is_wide_road"].sum()
si_mean = df_clean["safety_index"].mean()
print(f"  is_wide_road : 넓음(1)={wide:,}개 ({wide/len(df_clean)*100:.1f}%)")
print(f"  safety_index : 평균={si_mean:.3f}  결측={df_clean['safety_index'].isna().sum()}\n")

# 좌표 유무 구분
has_coord = df_clean[["start_lat", "start_lon"]].notna().all(axis=1)
print(f"  좌표 있는 행 : {has_coord.sum():,} / {len(df_clean):,} ({has_coord.mean()*100:.1f}%)")
print(f"  좌표 없는 행 : {(~has_coord).sum():,}  (기점지번주소 컬럼에 보존 → 추후 geocoding)\n")



STEP 4: 파생 피처 생성
  is_wide_road : 넓음(1)=362개 (7.2%)
  safety_index : 평균=0.100  결측=4635

  좌표 있는 행 : 1,751 / 4,998 (35.0%)
  좌표 없는 행 : 3,247  (기점지번주소 컬럼에 보존 → 추후 geocoding)



In [14]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: CSV 저장
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 5: 저장")
print("=" * 60)

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"  ✅ 저장 완료 → {OUTPUT_PATH}")
print(f"  최종 shape  : {df_clean.shape}")
print(f"  컬럼        : {list(df_clean.columns)}\n")



STEP 5: 저장
  ✅ 저장 완료 → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\data\raw_ml\road_clean_v2.csv
  최종 shape  : (4998, 14)
  컬럼        : ['route_name', 'sido', 'sigungu', 'start_addr_jibun', 'start_addr_road', 'start_lat', 'start_lon', 'end_lat', 'end_lon', 'length_km', 'road_id', 'width_m', 'is_wide_road', 'safety_index']



In [15]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: 모델 3개 비교 (좌표 있는 행만 사용)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 6: 모델 3개 비교 (타겟: safety_index)")
print("=" * 60)

df_model = df_clean[has_coord & df_clean["safety_index"].notna()].copy()
print(f"  모델 학습 대상 : {len(df_model):,}행\n")

if len(df_model) < 30:
    print("  ⚠️  학습 데이터 부족 (30행 미만) — 모델 비교 생략")
    sys.exit(0)

# 피처 / 타겟
FEATURES_SIMPLE = ["width_m", "length_km"]
FEATURES_MULTI  = ["width_m", "length_km", "start_lat", "start_lon"]
TARGET = "safety_index"

X_s = df_model[FEATURES_SIMPLE].fillna(df_model[FEATURES_SIMPLE].median())
X_m = df_model[FEATURES_MULTI].fillna(df_model[FEATURES_MULTI].median())
y   = df_model[TARGET]

X_s_tr, X_s_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, random_state=42)
X_m_tr, X_m_te, _,   _     = train_test_split(X_m, y, test_size=0.2, random_state=42)

results = {}


STEP 6: 모델 3개 비교 (타겟: safety_index)
  모델 학습 대상 : 57행



In [16]:

# ── 모델 1: LinearRegression (단순: 너비+길이) ──
lr = LinearRegression()
lr.fit(X_s_tr, y_tr)
y_pred_lr = lr.predict(X_s_te)
results["LinearRegression"] = {
    "R²" : round(r2_score(y_te, y_pred_lr), 4),
    "MSE": round(mean_squared_error(y_te, y_pred_lr), 4),
    "피처": FEATURES_SIMPLE,
}
print(f"  [1] LinearRegression     R²={results['LinearRegression']['R²']:.4f}")


  [1] LinearRegression     R²=1.0000


In [17]:

# ── 모델 2: PolynomialRegression (degree=2, 너비+길이) ──
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_tr = poly.fit_transform(X_s_tr)
X_poly_te = poly.transform(X_s_te)
lr_poly = LinearRegression()
lr_poly.fit(X_poly_tr, y_tr)
y_pred_poly = lr_poly.predict(X_poly_te)
results["PolynomialRegression(d=2)"] = {
    "R²" : round(r2_score(y_te, y_pred_poly), 4),
    "MSE": round(mean_squared_error(y_te, y_pred_poly), 4),
    "피처": FEATURES_SIMPLE + ["(degree=2 확장)"],
}
print(f"  [2] PolynomialRegression R²={results['PolynomialRegression(d=2)']['R²']:.4f}")


  [2] PolynomialRegression R²=1.0000


In [18]:

# ── 모델 3: RandomForestRegressor (너비+길이+위경도) ──
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_m_tr, y_tr)
y_pred_rf = rf.predict(X_m_te)
results["RandomForest"] = {
    "R²" : round(r2_score(y_te, y_pred_rf), 4),
    "MSE": round(mean_squared_error(y_te, y_pred_rf), 4),
    "피처": FEATURES_MULTI,
}
print(f"  [3] RandomForest         R²={results['RandomForest']['R²']:.4f}")


  [3] RandomForest         R²=0.8590


In [19]:

# 피처 중요도 출력
importances = rf.feature_importances_
print("\n  RandomForest 피처 중요도:")
for feat, imp in sorted(zip(FEATURES_MULTI, importances), key=lambda x: -x[1]):
    print(f"    {feat:<20}: {imp:.4f}")

# 결과 요약
print("\n  ── 모델 비교 요약 ──")
print(f"  {'모델':<30} {'R²':>8} {'MSE':>10}")
print(f"  {'-'*50}")
best_model = max(results, key=lambda k: results[k]["R²"])
for name, res in results.items():
    marker = " ← 최고" if name == best_model else ""
    print(f"  {name:<30} {res['R²']:>8.4f} {res['MSE']:>10.4f}{marker}")




  RandomForest 피처 중요도:
    width_m             : 0.4556
    start_lat           : 0.3415
    start_lon           : 0.1828
    length_km           : 0.0201

  ── 모델 비교 요약 ──
  모델                                   R²        MSE
  --------------------------------------------------
  LinearRegression                 1.0000     0.0000 ← 최고
  PolynomialRegression(d=2)        1.0000     0.0000
  RandomForest                     0.8590     0.0055


In [20]:

# ── 그래프 저장 ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
preds = [y_pred_lr, y_pred_poly, y_pred_rf]
titles = ["LinearRegression", "Polynomial(d=2)", "RandomForest"]

for ax, pred, title, res in zip(axes, preds, titles, results.values()):
    ax.scatter(y_te, pred, alpha=0.4, s=15)
    lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(f"{title}\nR²={res['R²']:.4f}")

plt.tight_layout()
plot_path = os.path.join(BASE_DIR, "data", "raw_ml", "model_comparison.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"\n  그래프 저장 → {plot_path}")
print("\n✅ preprocess_road.py 완료")



  그래프 저장 → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\data\raw_ml\model_comparison.png

✅ preprocess_road.py 완료


"""
build_safety_model.py
=====================
수도권 자전거도로 안전점수 모델 파이프라인

[ 실행 순서 ]
  1. python kride-project/preprocess_road.py   ← 먼저 실행 (road_clean_v2.csv 생성)
  2. python kride-project/build_safety_model.py ← 이 파일

[ 단계 요약 ]
  STEP 1 : road_clean_v2.csv 로드 (없으면 road_clean.csv fallback)
  STEP 2 : 사고다발지_서울.xlsx → 구별 위험도(district_danger) 계산
  STEP 3 : 도로 데이터 + 구별 위험도 병합 (sigungu 기준)
  STEP 4 : 피처 생성 + safety_index_v2 계산
             └ safety_index_v2 = (1 - district_danger) × 0.6 + road_attr_score × 0.4
  STEP 5 : 회귀 모델 학습 → safety_score 연속값 (0~1) 예측
  STEP 6 : 분류 모델 학습 → 위험등급 3단계 (0=안전/1=보통/2=위험) 예측
  STEP 7 : 평가 출력 (R², F1-macro)
  STEP 8 : 모델 저장

[ 출력 파일 ]
  kride-project/models/safety_regressor.pkl   — RandomForestRegressor
  kride-project/models/safety_classifier.pkl  — RandomForestClassifier
  kride-project/models/safety_scaler.pkl      — MinMaxScaler (추론 시 피처 정규화용)
  kride-project/data/raw_ml/district_danger.csv — 구별 위험도 테이블 (참고용)
"""


In [21]:

import os
import re
import sys
import warnings

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")


In [22]:

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# Jupyter에서는 __file__이 없으므로 fallback 처리
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Jupyter 실행 시: 현재 작업 디렉토리 기준 kride-project 폴더
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "kride-project"))
    if not os.path.exists(BASE_DIR):
        BASE_DIR = os.getcwd()  # kride-project 안에서 실행하는 경우
RAW_DIR     = os.path.join(BASE_DIR, "data", "raw_ml")
MODELS_DIR  = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

ROAD_V2_PATH    = os.path.join(RAW_DIR, "road_clean_v2.csv")
ROAD_V1_PATH    = os.path.join(RAW_DIR, "road_clean.csv")
ACCIDENT_PATH   = os.path.join(RAW_DIR, "다발지분석-24년 자전거 교통사고 다발지역_서울.xlsx")
DISTRICT_OUT    = os.path.join(RAW_DIR, "district_danger.csv")

REGRESSOR_PATH  = os.path.join(MODELS_DIR, "safety_regressor.pkl")
CLASSIFIER_PATH = os.path.join(MODELS_DIR, "safety_classifier.pkl")
SCALER_PATH     = os.path.join(MODELS_DIR, "safety_scaler.pkl")

RANDOM_STATE = 42



In [23]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: 도로 데이터 로드
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1: 도로 데이터 로드")
print("=" * 65)

if os.path.exists(ROAD_V2_PATH):
    df_road = pd.read_csv(ROAD_V2_PATH, encoding="utf-8-sig")
    print(f"  ✅ road_clean_v2.csv 로드 완료 — {df_road.shape}")
elif os.path.exists(ROAD_V1_PATH):
    df_road = pd.read_csv(ROAD_V1_PATH, encoding="utf-8-sig")
    print(f"  ⚠️  road_clean_v2.csv 없음 → road_clean.csv fallback — {df_road.shape}")
    print("     [ preprocess_road.py를 먼저 실행하면 더 많은 데이터를 사용할 수 있습니다 ]")
else:
    print("  ❌ 도로 CSV 파일을 찾을 수 없습니다. preprocess_road.py를 먼저 실행하세요.")
    sys.exit(1)


STEP 1: 도로 데이터 로드
  ✅ road_clean_v2.csv 로드 완료 — (4998, 14)


In [24]:

# sigungu 컬럼 확인 (v1은 한글, v2는 영문)
if "sigungu" in df_road.columns:
    sigungu_col = "sigungu"
elif "시군구명" in df_road.columns:
    df_road = df_road.rename(columns={"시군구명": "sigungu"})
    sigungu_col = "sigungu"
else:
    print("  ❌ sigungu 또는 시군구명 컬럼이 없습니다. 파일을 확인하세요.")
    sys.exit(1)

# width_m, length_km 컬럼 통일 (v1에서는 한글 컬럼명일 수 있음)
rename_map = {
    "자전거전용도로너비(m)": "width_m",
    "총길이(km)": "length_km",
    "기점위도": "start_lat",
    "기점경도": "start_lon",
}
df_road = df_road.rename(columns={k: v for k, v in rename_map.items() if k in df_road.columns})


In [25]:

# 필수 컬럼 숫자 변환
for col in ["width_m", "length_km", "start_lat", "start_lon"]:
    if col in df_road.columns:
        df_road[col] = pd.to_numeric(df_road[col], errors="coerce")

print(f"  컬럼 목록: {list(df_road.columns)}")
print(f"  시군구 목록 ({df_road[sigungu_col].nunique()}개): {sorted(df_road[sigungu_col].unique())[:10]} ...\n")



  컬럼 목록: ['route_name', 'sido', 'sigungu', 'start_addr_jibun', 'start_addr_road', 'start_lat', 'start_lon', 'end_lat', 'end_lon', 'length_km', 'road_id', 'width_m', 'is_wide_road', 'safety_index']
  시군구 목록 (30개): ['가평군', '강서구', '광주시', '구로구', '구리시', '군포시', '김포시', '남양주시', '노원구', '부천시'] ...



In [26]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: 사고다발지 데이터 → 구별 위험도 계산
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2: 사고다발지 데이터 → 구별 위험도(district_danger) 계산")
print("=" * 65)

if not os.path.exists(ACCIDENT_PATH):
    print(f"  ⚠️  사고다발지 파일 없음: {ACCIDENT_PATH}")
    print("     district_danger = 0 (전체 구 동일)으로 진행합니다.")
    df_danger = pd.DataFrame(columns=["sigungu", "danger_score", "district_danger"])
else:
    df_acc = pd.read_excel(ACCIDENT_PATH, engine="openpyxl")
    print(f"  사고다발지 shape: {df_acc.shape}")
    print(f"  컬럼: {list(df_acc.columns)}")
    print(f"  지점 예시:\n{df_acc.iloc[:5, 0].to_string()}\n")

    # 지점 컬럼에서 구 이름 추출 (예: "서울 중구1" → "중구")
    # 패턴: [가-힣]+(구|시|군) 중 구 우선
    jijum_col = df_acc.columns[0]  # 첫 번째 컬럼이 지점

    def extract_sigungu(text):
        text = str(text)
        # 먼저 구 이름 (예: 강남구, 중구)
        m = re.search(r"([가-힣]+구)", text)
        if m:
            return m.group(1)
        # 구 없으면 시/군 이름
        m = re.search(r"([가-힣]+(시|군))", text)
        if m:
            return m.group(1)
        return None


STEP 2: 사고다발지 데이터 → 구별 위험도(district_danger) 계산
  사고다발지 shape: (111, 8)
  컬럼: ['시도', '장소설명', '사고건수', '사상자수', '사망자수', '중상자수', '경상자수', '부상신고자수']
  지점 예시:
0     서울 중구1
1    서울 용산구1
2    서울 성동구1
3    서울 성동구2
4    서울 광진구1



In [27]:

    df_acc["sigungu"] = df_acc[jijum_col].apply(extract_sigungu)
    fail_cnt = df_acc["sigungu"].isna().sum()
    print(f"  구 이름 추출: 성공={len(df_acc) - fail_cnt}, 실패={fail_cnt}")
    print(f"  추출된 구 목록: {sorted(df_acc['sigungu'].dropna().unique())}\n")

    # 컬럼명 정규화 (실제 파일 컬럼명이 다를 수 있음)
    col_aliases = {
        "발생건수": ["발생건수", "사고건수"],
        "사망자수": ["사망자수", "사망자"],
        "중상자수": ["중상자수", "중상자"],
        "부상자수": ["부상자수", "부상자"],
    }

    def find_col(df, aliases):
        for a in aliases:
            if a in df.columns:
                return a
        return None

    c_occur = find_col(df_acc, col_aliases["발생건수"])
    c_dead  = find_col(df_acc, col_aliases["사망자수"])
    c_heavy = find_col(df_acc, col_aliases["중상자수"])
    c_injur = find_col(df_acc, col_aliases["부상자수"])

    print(f"  사용 컬럼: 발생건수={c_occur}, 사망자수={c_dead}, 중상자수={c_heavy}, 부상자수={c_injur}")


  구 이름 추출: 성공=111, 실패=0
  추출된 구 목록: ['강남구', '강동구', '강북구', '강서구', '광진구', '구로구', '금천구', '노원구', '동대문구', '동작구', '마포구', '성동구', '송파구', '양천구', '영등포구', '용산구', '은평구', '중구', '중랑구']

  사용 컬럼: 발생건수=사고건수, 사망자수=사망자수, 중상자수=중상자수, 부상자수=None


In [28]:

    # 위험도 가중 합산
    df_acc["danger_raw"] = 0.0
    if c_occur: df_acc["danger_raw"] += df_acc[c_occur].fillna(0) * 1.0
    if c_dead:  df_acc["danger_raw"] += df_acc[c_dead].fillna(0)  * 5.0
    if c_heavy: df_acc["danger_raw"] += df_acc[c_heavy].fillna(0) * 2.0
    if c_injur: df_acc["danger_raw"] += df_acc[c_injur].fillna(0) * 1.0

    # 구별 집계
    df_danger = (
        df_acc.dropna(subset=["sigungu"])
        .groupby("sigungu", as_index=False)["danger_raw"]
        .sum()
        .rename(columns={"danger_raw": "danger_score"})
    )

    # MinMaxScaler 정규화 (0=안전, 1=위험)
    scaler_d = MinMaxScaler()
    df_danger["district_danger"] = scaler_d.fit_transform(df_danger[["danger_score"]])

    df_danger.to_csv(DISTRICT_OUT, index=False, encoding="utf-8-sig")
    print(f"\n  구별 위험도 테이블 ({len(df_danger)}개 구):")
    print(df_danger.sort_values("district_danger", ascending=False).to_string(index=False))
    print(f"\n  저장 → {DISTRICT_OUT}")



  구별 위험도 테이블 (19개 구):
sigungu  danger_score  district_danger
    송파구         142.0         1.000000
   영등포구          87.0         0.601449
    중랑구          76.0         0.521739
    강동구          72.0         0.492754
   동대문구          64.0         0.434783
    구로구          54.0         0.362319
    양천구          48.0         0.318841
    은평구          42.0         0.275362
    강남구          40.0         0.260870
    강북구          39.0         0.253623
    노원구          26.0         0.159420
    강서구          24.0         0.144928
    마포구          24.0         0.144928
    동작구          15.0         0.079710
    성동구          14.0         0.072464
    금천구          13.0         0.065217
    용산구           8.0         0.028986
    광진구           6.0         0.014493
     중구           4.0         0.000000

  저장 → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\data\raw_ml\district_danger.csv


In [29]:


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: 도로 데이터 + 구별 위험도 병합
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3: 도로 + 구별 위험도 병합 (sigungu 기준)")
print("=" * 65)

if len(df_danger) > 0:
    danger_map = dict(zip(df_danger["sigungu"], df_danger["district_danger"]))
    df_road["district_danger"] = df_road[sigungu_col].map(danger_map)

    matched = df_road["district_danger"].notna().sum()
    total   = len(df_road)
    print(f"  위험도 매핑 성공: {matched:,}행 / 전체 {total:,}행 ({matched/total*100:.1f}%)")

    # 매핑 안 된 행(경기도 등) = 중앙값으로 대체
    median_danger = df_danger["district_danger"].median()
    df_road["district_danger"] = df_road["district_danger"].fillna(median_danger)
    print(f"  미매핑 행 → district_danger 중앙값 대체 ({median_danger:.4f})")
else:
    df_road["district_danger"] = 0.0
    print("  사고 데이터 없음 → district_danger = 0 (전체)")




STEP 3: 도로 + 구별 위험도 병합 (sigungu 기준)
  위험도 매핑 성공: 635행 / 전체 4,998행 (12.7%)
  미매핑 행 → district_danger 중앙값 대체 (0.2536)


In [30]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: 피처 생성 + safety_index_v2 계산
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4: 피처 생성 + safety_index_v2 계산")
print("=" * 65)

# 학습 대상: width_m, length_km 결측 없는 행
df_model = df_road.dropna(subset=["width_m", "length_km"]).copy()
print(f"  width_m/length_km 유효 행: {len(df_model):,}행 (전체 {len(df_road):,}행 중)")

# 도로 속성 정규화 (width × 0.7 + length × 0.3)
scaler_road = MinMaxScaler()
road_norm = scaler_road.fit_transform(df_model[["width_m", "length_km"]])
df_model["road_attr_score"] = road_norm[:, 0] * 0.7 + road_norm[:, 1] * 0.3

# 통합 안전지수 v2
# 위험도(district_danger)가 높을수록 unsafe → (1 - danger)로 반전
df_model["safety_index_v2"] = (
    (1 - df_model["district_danger"]) * 0.6
    + df_model["road_attr_score"] * 0.4
)

print(f"  safety_index_v2 통계:")
print(f"    평균={df_model['safety_index_v2'].mean():.4f}")
print(f"    최소={df_model['safety_index_v2'].min():.4f}")
print(f"    최대={df_model['safety_index_v2'].max():.4f}\n")



STEP 4: 피처 생성 + safety_index_v2 계산
  width_m/length_km 유효 행: 363행 (전체 4,998행 중)
  safety_index_v2 통계:
    평균=0.4831
    최소=0.4524
    최대=0.6632



In [31]:

# 위험등급 라벨 생성 (삼분위 기준)
# safety_index_v2 높을수록 안전 → 낮은 구간이 위험
q33 = df_model["safety_index_v2"].quantile(0.33)
q66 = df_model["safety_index_v2"].quantile(0.66)

def assign_danger_level(score):
    if score >= q66:
        return 0   # 안전
    elif score >= q33:
        return 1   # 보통
    else:
        return 2   # 위험

df_model["danger_level"] = df_model["safety_index_v2"].apply(assign_danger_level)
level_dist = df_model["danger_level"].value_counts().sort_index()
print(f"  위험등급 분포 (삼분위 기준: q33={q33:.4f}, q66={q66:.4f}):")
for lvl, cnt in level_dist.items():
    label = {0: "안전", 1: "보통", 2: "위험"}[lvl]
    print(f"    {lvl}({label}): {cnt:,}행 ({cnt/len(df_model)*100:.1f}%)")



  위험등급 분포 (삼분위 기준: q33=0.4661, q66=0.4787):
    0(안전): 124행 (34.2%)
    1(보통): 119행 (32.8%)
    2(위험): 120행 (33.1%)


In [32]:


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 & 6: 피처 / 타겟 준비 + 학습
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5+6: 모델 학습 (회귀 + 분류)")
print("=" * 65)

# 피처 선택
FEATURES = ["width_m", "length_km", "district_danger", "road_attr_score"]

# 위경도 있으면 추가 (공간 정보 보강)
if "start_lat" in df_model.columns and df_model["start_lat"].notna().sum() > 100:
    df_model_coord = df_model.dropna(subset=["start_lat", "start_lon"])
    FEATURES_WITH_COORD = FEATURES + ["start_lat", "start_lon"]
    print(f"  좌표 포함 행: {len(df_model_coord):,}행 → 좌표 포함 피처로 학습")
    X_df = df_model_coord[FEATURES_WITH_COORD].copy()
    y_reg = df_model_coord["safety_index_v2"].copy()
    y_cls = df_model_coord["danger_level"].copy()
    FINAL_FEATURES = FEATURES_WITH_COORD
else:
    X_df = df_model[FEATURES].copy()
    y_reg = df_model["safety_index_v2"].copy()
    y_cls = df_model["danger_level"].copy()
    FINAL_FEATURES = FEATURES
    print(f"  좌표 없음 → 기본 피처만 사용")



STEP 5+6: 모델 학습 (회귀 + 분류)
  좌표 없음 → 기본 피처만 사용


In [33]:

print(f"  최종 피처: {FINAL_FEATURES}")
print(f"  학습 대상: {len(X_df):,}행\n")


  최종 피처: ['width_m', 'length_km', 'district_danger', 'road_attr_score']
  학습 대상: 363행



In [34]:

if len(X_df) < 30:
    print("  ❌ 학습 데이터 부족 (30행 미만). 데이터 확인 후 재실행하세요.")
    sys.exit(1)

# 피처 스케일링
scaler_feat = MinMaxScaler()
X_scaled = scaler_feat.fit_transform(X_df)

X_tr, X_te, y_reg_tr, y_reg_te, y_cls_tr, y_cls_te = train_test_split(
    X_scaled, y_reg, y_cls,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_cls,     # 분류 비율 유지
)

print(f"  학습: {len(X_tr):,}행 / 테스트: {len(X_te):,}행")


  학습: 290행 / 테스트: 73행


In [35]:

# ── 회귀 모델 ────────────────────────────────────────────────────────────────
print("\n  [ 회귀: RandomForestRegressor ]")
rf_reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_reg.fit(X_tr, y_reg_tr)
y_pred_reg = rf_reg.predict(X_te)

r2  = r2_score(y_reg_te, y_pred_reg)
mse = mean_squared_error(y_reg_te, y_pred_reg)
print(f"  R²  = {r2:.4f}")
print(f"  MSE = {mse:.6f}")

print("\n  피처 중요도 (회귀):")
for feat, imp in sorted(zip(FINAL_FEATURES, rf_reg.feature_importances_), key=lambda x: -x[1]):
    print(f"    {feat:<22}: {imp:.4f}")




  [ 회귀: RandomForestRegressor ]
  R²  = 0.9539
  MSE = 0.000055

  피처 중요도 (회귀):
    road_attr_score       : 0.9403
    width_m               : 0.0507
    length_km             : 0.0089
    district_danger       : 0.0000


In [36]:

# ── 분류 모델 ────────────────────────────────────────────────────────────────
print("\n  [ 분류: RandomForestClassifier ]")
rf_cls = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",   # 클래스 불균형 대응
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_cls.fit(X_tr, y_cls_tr)
y_pred_cls = rf_cls.predict(X_te)

f1 = f1_score(y_cls_te, y_pred_cls, average="macro")
print(f"  F1-macro = {f1:.4f}")
print("\n  분류 리포트:")
print(classification_report(y_cls_te, y_pred_cls, target_names=["안전(0)", "보통(1)", "위험(2)"]))



  [ 분류: RandomForestClassifier ]
  F1-macro = 0.9864

  분류 리포트:
              precision    recall  f1-score   support

       안전(0)       0.96      1.00      0.98        25
       보통(1)       1.00      1.00      1.00        24
       위험(2)       1.00      0.96      0.98        24

    accuracy                           0.99        73
   macro avg       0.99      0.99      0.99        73
weighted avg       0.99      0.99      0.99        73



In [37]:

print("\n  피처 중요도 (분류):")
for feat, imp in sorted(zip(FINAL_FEATURES, rf_cls.feature_importances_), key=lambda x: -x[1]):
    print(f"    {feat:<22}: {imp:.4f}")




  피처 중요도 (분류):
    road_attr_score       : 0.6586
    length_km             : 0.2139
    width_m               : 0.1205
    district_danger       : 0.0071


In [38]:

# ── 분류 모델 ────────────────────────────────────────────────────────────────
print("\n  [ 분류: RandomForestClassifier ]")
rf_cls = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",   # 클래스 불균형 대응
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_cls.fit(X_tr, y_cls_tr)
y_pred_cls = rf_cls.predict(X_te)

f1 = f1_score(y_cls_te, y_pred_cls, average="macro")
print(f"  F1-macro = {f1:.4f}")
print("\n  분류 리포트:")
print(classification_report(y_cls_te, y_pred_cls, target_names=["안전(0)", "보통(1)", "위험(2)"]))

print("\n  피처 중요도 (분류):")
for feat, imp in sorted(zip(FINAL_FEATURES, rf_cls.feature_importances_), key=lambda x: -x[1]):
    print(f"    {feat:<22}: {imp:.4f}")




  [ 분류: RandomForestClassifier ]
  F1-macro = 0.9864

  분류 리포트:
              precision    recall  f1-score   support

       안전(0)       0.96      1.00      0.98        25
       보통(1)       1.00      1.00      1.00        24
       위험(2)       1.00      0.96      0.98        24

    accuracy                           0.99        73
   macro avg       0.99      0.99      0.99        73
weighted avg       0.99      0.99      0.99        73


  피처 중요도 (분류):
    road_attr_score       : 0.6586
    length_km             : 0.2139
    width_m               : 0.1205
    district_danger       : 0.0071


In [39]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7: 추론 인터페이스 확인 (샘플 테스트)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7: 샘플 추론 테스트")
print("=" * 65)

# 추론 시 입력 예시
sample_input = {
    "width_m"         : 3.0,    # 너비 3m
    "length_km"       : 5.0,    # 길이 5km
    "district_danger" : 0.3,    # 구 위험도 30%
    "road_attr_score" : 0.6,    # 도로 속성 60점
}
# 위경도 포함 피처인 경우 추가
if "start_lat" in FINAL_FEATURES:
    sample_input["start_lat"] = 37.55
    sample_input["start_lon"] = 127.05

sample_df  = pd.DataFrame([[sample_input[f] for f in FINAL_FEATURES]], columns=FINAL_FEATURES)
sample_scaled = scaler_feat.transform(sample_df)

pred_score = rf_reg.predict(sample_scaled)[0]
pred_level = rf_cls.predict(sample_scaled)[0]
level_label = {0: "안전", 1: "보통", 2: "위험"}[pred_level]

print(f"  입력: width_m=3.0, length_km=5.0, district_danger=0.3")
print(f"  → safety_score (회귀): {pred_score:.4f}")
print(f"  → 위험등급  (분류): {pred_level} ({level_label})")




STEP 7: 샘플 추론 테스트
  입력: width_m=3.0, length_km=5.0, district_danger=0.3
  → safety_score (회귀): 0.6061
  → 위험등급  (분류): 0 (안전)


In [40]:

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8: 모델 저장
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8: 모델 저장")
print("=" * 65)

joblib.dump(rf_reg,      REGRESSOR_PATH)
joblib.dump(rf_cls,      CLASSIFIER_PATH)
joblib.dump(scaler_feat, SCALER_PATH)

# 추론 시 필요한 메타 정보도 함께 저장
meta = {
    "features"    : FINAL_FEATURES,
    "danger_level": {0: "안전", 1: "보통", 2: "위험"},
    "q33"         : q33,
    "q66"         : q66,
    "r2_regressor": round(r2, 4),
    "f1_classifier": round(f1, 4),
}
joblib.dump(meta, os.path.join(MODELS_DIR, "safety_meta.pkl"))



STEP 8: 모델 저장


['c:\\Users\\human-32\\OneDrive\\ドキュメント\\yerinMin\\humaneducation\\kride-project\\models\\safety_meta.pkl']

In [41]:

print(f"  ✅ safety_regressor.pkl  → {REGRESSOR_PATH}")
print(f"  ✅ safety_classifier.pkl → {CLASSIFIER_PATH}")
print(f"  ✅ safety_scaler.pkl     → {SCALER_PATH}")
print(f"  ✅ safety_meta.pkl       → {os.path.join(MODELS_DIR, 'safety_meta.pkl')}")
print(f"  ✅ district_danger.csv   → {DISTRICT_OUT}")

print("\n" + "=" * 65)
print("✅ build_safety_model.py 완료")
print(f"   회귀 R²      : {r2:.4f}")
print(f"   분류 F1-macro: {f1:.4f}")
print("=" * 65)

  ✅ safety_regressor.pkl  → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\models\safety_regressor.pkl
  ✅ safety_classifier.pkl → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\models\safety_classifier.pkl
  ✅ safety_scaler.pkl     → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\models\safety_scaler.pkl
  ✅ safety_meta.pkl       → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\models\safety_meta.pkl
  ✅ district_danger.csv   → c:\Users\human-32\OneDrive\ドキュメント\yerinMin\humaneducation\kride-project\data\raw_ml\district_danger.csv

✅ build_safety_model.py 완료
   회귀 R²      : 0.9539
   분류 F1-macro: 0.9864


d_tourism_model.py
======================
수도권 자전거도로 관광점수 모델 파이프라인

[ 실행 순서 ]
  1. python kride-project/preprocess_road.py      ← road_clean_v2.csv 생성
  2. python kride-project/build_safety_model.py   ← safety pkl 생성
  3. python kride-project/build_tourism_model.py  ← 이 파일

[ 단계 요약 ]
  STEP 1 : road_features.csv 로드 (tourist_count, cultural_count 등)
  STEP 2 : 관광점수(tourism_score) 계산
             └ raw = tourist_count×0.5 + cultural_count×0.3 + leisure_count×0.2
             └ tourism_score = MinMaxScaler(raw) + facility_bonus (cap 0.1)
  STEP 3 : safety_score 계산 (safety_regressor.pkl 적용)
  STEP 4 : final_score = safety_score×0.6 + tourism_score×0.4
  STEP 5 : 결과 저장

[ 출력 파일 ]
  kride-project/models/tourism_scaler.pkl      — MinMaxScaler (관광 raw score용)
  kride-project/data/raw_ml/road_scored.csv   — 전체 점수 포함 최종 데이터셋
"""